# Morning Star：用 qust 识别早晨之星

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


来源参考：[Investopedia](https://www.investopedia.com/terms/m/morningstar.asp)

本文按 Investopedia 原文结构讲解指标含义、常见用法和局限性，并展示如何用 qust 一行计算指标、选择单个 `ticker + ct` 合约画图，以及在完整多合约数据上按 `over("ticker", "ct")` 做回测。


## 1. Investopedia 原文内容完整改写：Morning Star

### 什么是 Morning Star
Morning Star 是三根 K 线组成的看涨反转形态。名字暗示“夜晚之后出现晨星”：市场先经历一根明显下跌蜡烛，随后出现一根小实体表示犹豫，最后第三根强阳线把价格重新推高。它通常出现在下跌趋势末端。

### 三根 K 线结构
第一根是较长阴线，代表空头仍在推动价格下行。第二根是小实体，可以是小阴、小阳或十字星，代表市场犹豫和动能减弱。第三根是阳线，最好能明显收回第一根阴线实体的一部分，说明买方重新取得主动。

### 市场心理
第一根让下跌看起来延续；第二根没有继续大幅走弱，说明卖方力量开始衰减；第三根强势上涨则改变了短期预期。这个三段式变化比单根 K 线更强调情绪转换过程。

### 确认和使用
交易者通常关注第三根阳线的力度，例如是否收回第一根实体的一半以上，是否放量，是否突破第二根高点，是否发生在支撑区域。形态确认后，风险常参考第二根或整个形态的低点。

### 参数和变体
不同软件对第二根“小实体”和第三根“收回多少”定义不同。有些市场存在跳空，传统蜡烛书会强调第二根与第一根之间的 gap；但在连续交易或期货数据中 gap 不总是稳定出现，因此程序化常用实体大小和收复比例替代。

### 局限性
Morning Star 可能出现后价格只短暂反弹。它需要趋势背景和后续确认，否则只是普通三根 K 线组合。强下跌趋势中，任何看涨反转形态都可能失败。

## 2. 从文章到 qust 算子的落地

qust 用 `small_body_ratio` 控制第二根实体大小，用 `close_into_body` 控制第三根收复第一根实体的比例。默认还检查下跌背景，输出布尔列 `morning_star`。

## 3. qust 一行调用

```python
col("open", "high", "low", "close").investopedia.morning_star()
```

输入列顺序：`open, high, low, close`。

输出列：`morning_star`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实GitHub K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("open", "high", "low", "close").investopedia.morning_star()
morning_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    morning_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("morning_star").cast(pl.UInt32).sum().alias("morning_star_count"),
).calc_data(morning_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 9)


morning_star_count
u32
1264


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 Jupyter 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
morning_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("morning_price", show_axis_label=True)
        .kline(),
    col("datetime", "low", "morning_star")
        .monitor("morning_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["morning_price"],
]).runtime()

morning_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Morning Star 策略回测

早晨之星是看涨反转形态，但默认严格参数信号太少、收益不稳定。这里放宽形态参数：不额外要求前置趋势，允许第二根小实体比例到 1.0，并要求第三根收回第一根实体 70%。交易方向仍按文章含义做多；使用 5% 止盈、2% 止损，并对 `hold` 做 `fp.vol_pms` 归一化。

In [4]:
TAKE_PROFIT = 0.05
STOP_LOSS = 0.02

indicator_cols = col("open", "high", "low", "close").investopedia.morning_star(
    trend_period=0,
    small_body_ratio=1.0,
    close_into_body=0.7,
)
strategy_daily_expr = (
    col
    .with_cols(indicator_cols)
    .with_cols(
        (col("morning_star")).fill_null(col.lit(False)).alias("open_long_raw"),
        (col.lit(False)).fill_null(col.lit(False)).alias("open_short_raw"),
    )
    # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
    .with_cols(
        col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
        col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
        col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
        col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
        col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
    )
    .with_cols(
        (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
            .fill_null(col.lit(False))
            .alias("exit_long_sig"),
        (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
            .fill_null(col.lit(False))
            .alias("exit_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols((col("hold") / col.all.fp.vol_pms()).alias("hold"))
    .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
    .over("ticker", "ct")
    .select(
        col("pnl")
            .sum()
            .group_by(col("datetime").dt.date().alias("date"))
            .batch.sort("date")
            .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
            .select("date", "pnl", "pnl_cum")
    )
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = col("date", "pnl").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


metric,value,value_float
str,str,f64
"""Start Index""","""2022-01-04""",null
"""End Index""","""2024-12-31""",null
"""Total Duration""","""1092 days, 0:00:00""",null
"""Total Return [%]""","""3.815329248753348e+48""",3.8153e48
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""","""5774099388076225.0""",5.7741e15
"""Annualized Volatility [%]""","""2937.952224554212""",2937.952225
"""Max Drawdown [%]""","""221978.1898581617""",221978.189858
…,…,…


In [5]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,0.874114,53.321088
2024-12-19,-1.651979,51.66911
2024-12-20,0.323268,51.992378
2024-12-21,-0.125345,51.867032
2024-12-23,0.475886,52.342918
2024-12-24,1.23771,53.580628
2024-12-25,-0.697574,52.883054
2024-12-26,-0.359509,52.523544
2024-12-27,-2.347694,50.17585


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。